In [ ]:
import pandas as pd
import io

pd.options.display.max_rows = 200

In [ ]:
def _process_data_(file):
    with open(file, 'r') as file:
        log_content = file.read()
    sections = log_content.split('Sandbox logs:')[1].split('Activities log:')
    sandbox_log =  sections[0].strip()
    activities_log = sections[1].split('Trade History:')[0]
    # sandbox_log_list = [json.loads(line) for line in sandbox_log.split('\n')]
    trade_history =  json.loads(sections[1].split('Trade History:')[1])
    # sandbox_log_df = pd.DataFrame(sandbox_log_list)
    market_data_df = pd.read_csv(io.StringIO(activities_log), sep=";", header=0)
    trade_history_df = pd.json_normalize(trade_history)
    return market_data_df, trade_history_df

In [ ]:
df_past = pd.read_csv('r4.csv',sep=';')

In [ ]:
df_past.head(200)

In [ ]:
df_market, _ = _process_data_('r4_results.log')

In [ ]:
df_market.head(200)

In [ ]:
df_past = df_past.pivot(columns='product', values='mid_price', index='timestamp').reset_index()

In [ ]:
df_past.columns

In [ ]:
df_market = df_market.pivot(columns='product', values='mid_price', index='timestamp').reset_index()

In [ ]:
df_market.columns

In [ ]:
df_coconuts = df_market[['timestamp','COCONUT']].merge(df_past[['timestamp','COCONUTS']], on='timestamp', how='inner')

In [ ]:
import plotly.express as px

# Assuming df_past and df_market have a common 'timestamp' column
fig = px.line(df_past, x='timestamp', y=['COCONUTS'])

# Add the trace for df_market's 'COCONUT' column
fig.add_scatter(x=df_market['timestamp'], y=df_market['COCONUT'], name='COCONUT (Market)')

# Set the y-axis labels
fig.update_layout(
    yaxis=dict(title='COCONUTS (Past)'),
    yaxis2=dict(title='COCONUT (Market)', overlaying='y', side='right')
)

# Show the plot
fig.show()

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Create a subplot with multiple y-axes
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Add traces for df_past
for col in df_past.columns[1:]:
    fig.add_trace(go.Scatter(x=df_past['timestamp'], y=df_past[col], name=col), secondary_y=False)

# Add traces for df_market
for col in df_market.columns[1:]:
    fig.add_trace(go.Scatter(x=df_market['timestamp'], y=df_market[col], name=col), secondary_y=True)

# Update layout
fig.update_layout(
    title='Correlation Plot',
    xaxis=dict(title='Timestamp'),
    yaxis=dict(title='df_past'),
    yaxis2=dict(title='df_market', overlaying='y', side='right'),
    width=1200,  # Adjust the width as needed
    height=800   # Adjust the height as needed
)

fig.update_layout(autosize=True) # remove height=800
fig.show(renderer="browser")

In [ ]:
import plotly.graph_objects as go

# Create traces for df_past
traces = []
for i, col in enumerate(df_past.columns[1:]):
    trace = go.Scatter(x=df_past['timestamp'], y=df_past[col], name=col, yaxis=f'y{i+1}')
    traces.append(trace)

# Create traces for df_market
for i, col in enumerate(df_market.columns[1:], start=len(df_past.columns[1:])):
    trace = go.Scatter(x=df_market['timestamp'], y=df_market[col], name=col, yaxis=f'y{i+1}')
    traces.append(trace)

# Create the layout for the graph
layout = go.Layout(
    title='Correlation Plot',
    xaxis=dict(title='Timestamp'),
    width=2500,  # Adjust the width as needed
    height=2000  # Adjust the height as needed
)

# Update y-axis properties for df_past
for i, col in enumerate(df_past.columns[1:]):
    layout[f'yaxis{i+1}'] = dict(title=col, side='left', overlaying='y', position=0.05 + i*0.02)

# Update y-axis properties for df_market
for i, col in enumerate(df_market.columns[1:], start=len(df_past.columns[1:])):
    layout[f'yaxis{i+1}'] = dict(title=col, side='right', overlaying='y', position=0.95 - (i - len(df_past.columns[1:])) * 0.02)

# Create the figure and add the traces and layout
fig = go.Figure(data=traces, layout=layout)

fig.update_layout(autosize=True)  # remove height=800

fig.show(renderer="browser")

In [ ]:
import plotly.graph_objects as go

# Create traces for df_past
traces = []
for i, col in enumerate(df_past.columns[1:]):
    trace = go.Scatter(x=df_past['timestamp'], y=df_past[col], name=col, yaxis=f'y{i+1}')
    traces.append(trace)

# Create traces for df_market
for i, col in enumerate(df_market.columns[1:], start=len(df_past.columns[1:])):
    trace = go.Scatter(x=df_market['timestamp'], y=df_market[col], name=col, yaxis=f'y{i+1}')
    traces.append(trace)

# Add a dummy trace on yaxis1 to ensure it's always drawn
dummy_trace = go.Scatter(x=[0], y=[0], visible=False)
traces.insert(0, dummy_trace)

# Create the layout for the graph
layout = go.Layout(
    title='Correlation Plot',
    xaxis=dict(title='Timestamp'),
    width=2500,  # Adjust the width as needed
    height=2000  # Adjust the height as needed
)

# Update y-axis properties for df_past
for i, col in enumerate(df_past.columns[1:]):
    layout[f'yaxis{i+1}'] = dict(title=col, side='left', overlaying='y' if i > 0 else None, position=0.05 + i*0.02)

# Update y-axis properties for df_market
for i, col in enumerate(df_market.columns[1:], start=len(df_past.columns[1:])):
    layout[f'yaxis{i+1}'] = dict(title=col, side='right', overlaying=f'y{i}' if i > len(df_past.columns[1:]) else None, position=0.95 - (i - len(df_past.columns[1:])) * 0.02)

# Create the figure and add the traces and layout
fig = go.Figure(data=traces, layout=layout)
fig.update_layout(autosize=True)  # remove height=800

fig.show(renderer="browser")

In [ ]:
df_past

In [ ]:
df_market

In [ ]:
df_market['IMPLIED_ROSES'] = df_market['GIFT_BASKET'] - 6* df_market['STRAWBERRIES'] - 4*df_market['CHOCOLATE']

In [ ]:
df_market

In [ ]:
def get_prev_returns(df, col, its):
    prev_col = f"{col}_prev_{its}_its"
    df[prev_col] = df[col].shift(its)
    df[f"{col}_returns_from_{its}_its_ago"] = (df[col] - df[prev_col]) / df[prev_col]
    df.drop(columns=[prev_col], inplace=True)
    return df

def get_future_returns(df, col, its):
    future_col = f"{col}_future_{its}_its"
    df[future_col] = df[col].shift(-its)
    df[f"{col}_returns_in_{its}_its"] = (df[future_col] - df[col]) / df[col]
    df.drop(columns=[future_col], inplace=True)
    return df

def get_centered_returns(df, col, its):
    future_col = f"{col}_future_{its}_its"
    df[future_col] = df[col].shift(-its)
    prev_col = f"{col}_prev_{its}_its"
    df[prev_col] = df[col].shift(its)
    df[f"{col}_returns_centered_with_{its}_its"] = (df[future_col] - df[prev_col])/df[prev_col]
    df.drop(columns=[prev_col], inplace=True)
    df.drop(columns=[future_col], inplace=True)
    return df

In [ ]:
df_implied_roses = df_market[['timestamp','IMPLIED_ROSES']].merge(df_past[['timestamp', 'DIVING_GEAR']], on='timestamp', how='inner')

In [ ]:
iterations = [1,5,10,20]

df_test = df_implied_roses.copy()
for it in iterations: 
    df_test = get_future_returns(df_test, 'IMPLIED_ROSES', it)
    df_test = get_future_returns(df_test, 'DIVING_GEAR', it)

In [ ]:
df_test

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import scipy.stats as stats


predictor_timeframes = [1,5,10,15,20]
responder_timeframes = [1,5,10,15,20]
predictor_symbols = ['DIVING_GEAR']
responder_symbols = ['IMPLIED_ROSES']

df_copy = df_implied_roses.copy()

for responder_timeframe in responder_timeframes:
    print(f"Responder Timeframe: {responder_timeframe} iterations")
    
    # Add future returns columns for responder symbols
    for symbol in responder_symbols:
        df_copy = get_future_returns(df_copy, symbol, responder_timeframe)
    
    for predictor_timeframe in predictor_timeframes:
        print(f"Predictor Timeframe: {predictor_timeframe} iterations")
        
        # Add lagged returns columns for predictor symbols
        for symbol in predictor_symbols:
            df_copy = get_future_returns(df_copy, symbol, predictor_timeframe)
        
        for target_symbol in responder_symbols:
            print(f"Target Symbol: {target_symbol}")
            
            # Get the feature columns (lagged returns from predictor symbols)
            feature_cols = [col for col in df_copy.columns if col.endswith(f"_returns_in_{predictor_timeframe}_its") and any(col.startswith(symbol) for symbol in predictor_symbols)]
            
            # Get the target column (future returns for the target symbol)
            target_col = f"{target_symbol}_returns_in_{responder_timeframe}_its"
            
            # Drop rows with missing values
            df_train = df_copy[feature_cols + [target_col]].dropna()
            
            # Split the data into features (X) and target (y)
            X = df_train[feature_cols]
            y = df_train[target_col]
            
            # Create and fit the linear regression model (without y-intercept)
            model = LinearRegression(fit_intercept=False)
            model.fit(X, y)
            
            # Print the learned equation
            equation = f"{target_col} = " + " + ".join([f"{coef:.4f} * {feat}" for feat, coef in zip(feature_cols, model.coef_)])
            print("Learned Equation:")
            print(equation)
            
            # Make predictions on the training data
            y_pred = model.predict(X)
            
            # Calculate and print the R-squared and p-value
            r2 = r2_score(y, y_pred)
            print(f"R-squared: {r2:.4f}")
            
            _, p_value = stats.pearsonr(y, y_pred)
            print(f"p-value: {p_value:.4f}")
            
            print()
        
        # Remove lagged returns columns for predictor symbols
        lagged_cols = [col for col in df_copy.columns if col.endswith(f"_returns_from_{predictor_timeframe}_its_ago")]
        df_copy.drop(columns=lagged_cols, inplace=True)
    
    # Remove future returns columns for responder symbols
    future_cols = [col for col in df_copy.columns if col.endswith(f"_returns_in_{responder_timeframe}_its")]
    df_copy.drop(columns=future_cols, inplace=True)
    
    print()

```
Predictor Timeframe: 10 iterations
Target Symbol: IMPLIED_ROSES
Learned Equation:
IMPLIED_ROSES_returns_in_10_its = 3.1835 * DIVING_GEAR_returns_in_10_its
R-squared: 0.1920
p-value: 0.0000
```

In [ ]:
df_test = df_implied_roses.copy()

In [ ]:
df_test = get_future_returns(df_test, 'DIVING_GEAR', 10)

In [ ]:
df_test['DIVING_GEAR_returns_in_10_its'].abs().describe()